# Class 8: Building Agents with n8n & Zapier

**Tools:** n8n (cloud at app.n8n.cloud -- free trial), Zapier (zapier.com -- free tier), Google account (for Docs/Gmail/Calendar)  

### Agenda:
1. How to wire an **n8n AI Agent node** to real tools -- web search, Gmail, Google Calendar, databases -- and let the LLM decide which tool to call
2. A clear understanding of **what counts as a tool** for an agent and how the agent autonomously chooses which one to invoke
3. How to use the **Zapier AI agent feature** for simpler trigger-based automations
4. Practical **error handling and retry logic** strategies that make agentic loops reliable in production
5. A working **no-code research agent** you built yourself: takes a topic, searches the web, summarises findings, and saves to a Google Doc

## Section 1 - Intro: From Paper Specs to Working Agents (10 min)

> "Last class you learned what an agent IS -- the Think-Act-Observe loop, tools, memory, autonomy. You even designed agent specs on paper. Today we build them for real.
>
> But here's the important part: **we're not writing code**. We're using no-code platforms -- n8n and Zapier -- that let you visually wire together agent workflows. Why?
>
> Because the hardest part of building agents isn't the code. It's the **architecture decisions**: which tools to give the agent, how to handle failures, when to let it run autonomously vs when to require human approval. No-code platforms let you focus on those decisions without getting lost in boilerplate.
>
> Think of today as learning to drive before learning how the engine works. In later classes we'll build agents in Python. But the design thinking you develop today transfers directly."



**Why n8n and Zapier specifically?**

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/191/846/original/1.png?1776767450)


**Why this matters for SWEs:**
- n8n is open-source and self-hostable -- many engineering teams run it internally for AI-powered automations
- Zapier's AI features are used by product teams to ship agent-powered workflows without engineering sprints
- Understanding no-code agent patterns makes you faster at designing code-based agents later -- same architecture, different implementation

## Section 2 -- Concept in 5 Minutes: What Counts as a Tool?

> "Before we build anything, let's nail down the concept that makes agents work: **tools**. In Class 7, I said tools are the agent's 'hands.' Today let's be precise about what that means."

### A tool is any function the LLM can call to interact with the outside world.

That's it. A tool takes an input, does something, and returns an output. The LLM doesn't execute the tool itself -- it **decides** to call it, the platform **executes** it, and the result gets fed **back** to the LLM.

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/191/849/original/2.png?1776767775)


**Key insight:** The LLM never directly accesses the internet, sends emails, or writes to databases. It **asks the platform to do it**. The platform is the gatekeeper -- which means you control what the agent can and cannot do by choosing which tools to connect.

### Examples of tools across platforms:

```
┌──────────────────┬─────────────────────────────────────────┐
│ Tool category    │ Concrete examples                       │
├──────────────────┼─────────────────────────────────────────┤
│ Information      │ Web search, Wikipedia lookup, API call  │
│ Communication    │ Send email, post to Slack, send SMS     │
│ File operations  │ Read/write Google Docs, create PDF      │
│ Data             │ Query database, read spreadsheet        │
│ Calendar         │ Create event, check availability        │
│ Code execution   │ Run Python, execute SQL query           │
│ Custom           │ Any API endpoint wrapped as a tool      │
└──────────────────┴─────────────────────────────────────────┘
```

**Beginner Note:** Think of tools like apps on your phone. The agent (your brain) decides which app to open. The app (tool) does the actual work.

## Section 3 -- Tool Demo: The n8n AI Agent Node (15 min)

> "Let me show you the n8n AI Agent node. This is where last class's diagrams become real. You'll literally see the Think-Act-Observe loop on a visual canvas."

### What is n8n?

n8n is an open-source workflow automation platform. You build workflows by dragging nodes onto a canvas and connecting them. It has 400+ integrations (Gmail, Slack, databases, APIs) and -- crucially -- a dedicated **AI Agent node** that implements the ReAct loop natively.

**Setup:** Go to [app.n8n.cloud](https://app.n8n.cloud/) and create a free account (no credit card needed for trial).

### Step-by-step demo (share scren):

**Step 1 -- Create a new workflow.**  
Click **"+ Add workflow"** from the dashboard. You get a blank canvas with a trigger node.

**Step 2 -- Add a trigger.**  
For our demo, click the trigger node and select **"Chat Trigger"** (under AI category). This lets us interact with the agent directly in n8n's built-in chat interface.

**Step 3 -- Add the AI Agent node.**  
Click the **+** button after the trigger. Search for **"AI Agent"** and add it. This is the ReAct loop engine.

**Step 4 -- Configure the Agent's LLM brain.**  
Click on the AI Agent node. You'll see connection points at the bottom for:
- **Chat Model** (required) -- the LLM brain
- **Tools** (this is where the power comes from)
- **Memory** (optional but recommended)

Click **Chat Model** and add an **OpenAI Chat Model** node (or Google Gemini). Enter your API key.

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/192/798/original/Pi7_image_tool.jpeg?1777269426)

**Step 5 -- Add tools.**  
Click the **Tools** connector on the Agent node. This is where you give the agent its capabilities. Add these one at a time:

**Tool 1: Web Search (SerpAPI or built-in HTTP Request)**
- Search for **"SerpAPI"** in the tools panel and add it
- This gives the agent the ability to search the web
- The agent will see this tool described as: *"Search the web for information on any topic"*

**Tool 2: Google Docs (create/update documents)**
- Search for **"Google Docs"** and add the create-document tool
- Connect your Google account when prompted
- The agent will see this tool as: *"Create a new Google Doc with a title and content"*

**Step 6 -- Add memory.**  
Click the **Memory** connector and add **"Window Buffer Memory"**. Set the window size to 10 messages. This lets the agent remember earlier steps in the conversation.

**Step 7 -- Set the system prompt.**  
In the AI Agent node settings, find the **"System Message"** field. This is the agent's standing instructions -- exactly like the system prompts from Class 3, but now the agent can ACT on them.

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/192/799/original/2.jpeg?1777270153)

**Step 8 -- Test it.**  
Click **"Chat"** in the top bar to open the built-in chat panel. Type:

```
Research the current state of WebAssembly adoption in 2026.
Save your findings to a Google Doc.
```

**What to watch for (map it to the ReAct loop):**

```
THINK  --> "I need to search for WebAssembly adoption in 2026"
ACT    --> [Calls Web Search tool with query]
OBSERVE--> Gets search results
THINK  --> "I have enough info to write a summary. Let me create the doc."
ACT    --> [Calls Google Docs tool to create document]
OBSERVE--> Document created successfully
DONE   --> "I've researched WebAssembly adoption and saved a summary
            to a Google Doc titled '...'. Here's what I found: ..."
```

> "Look at the execution panel on the right. n8n shows you *every* step the agent took -- which tools it called, what inputs it sent, what results came back. This is the ReAct loop made visible. In code-based agents you'd have to build this logging yourself."

**Beginner Note:** Everything you just saw happened because you connected nodes on a canvas. No code was written.

## Section 4 -- Hands-On Usage (25 min)

### Part A: Build Your n8n Research Agent (15 min)

**Your turn. Replicate what you just saw, then extend it.**

### Exercise 4.1 -- Build the Basic Research Agent

**Open [app.n8n.cloud](https://app.n8n.cloud/) and build this workflow:**

```
YOUR WORKFLOW:

  [Chat Trigger] --> [AI Agent] --> (output in chat)
                        │
                  Connected:
                  ├── Chat Model: OpenAI GPT-4o or Gemini
                  ├── Tool: Web Search (SerpAPI or HTTP Request)
                  ├── Tool: Google Docs (create document)
                  └── Memory: Window Buffer (10 messages)
```

**System message:** Use the one from the demo (or write your own -- the key elements are: search, summarise, save to doc, cite sources).

**Test with these prompts (one at a time):**

1. `Research the top 3 AI coding assistants in 2026 and compare their features. Save to a Google Doc.`

2. `What are the biggest changes in the Python 3.14 release? Save a summary to Google Docs.`



**What success looks like:**
- The agent searches the web (you see the tool call in the execution log)
- It writes a structured summary (not just raw search results)
- A Google Doc is actually created in your Drive with the content
- The agent tells you what it found and where it saved it

**Troubleshooting:**
- If the agent doesn't search: check that the web search tool is properly connected to the Agent node's Tools input
- If Google Docs fails: make sure you've authenticated your Google account in the Google Docs node
- If the output is messy: improve your system message with more specific formatting instructions

**Beginner Note:** Take it step by step. Get the Chat Trigger + AI Agent + Chat Model working first (text-only). Then add the web search tool. Then add Google Docs. Test after each addition.

**Intermediate Note:** Check the execution panel after each run. You can see exactly which tools the agent called and in what order. This is your debugging superpower.

### Exercise 4.2 -- Extend with More Tools

**Add ONE more tool to your agent. Choose based on what's useful to you:**

| Tool to add | What it enables | How to find it in n8n |
|:---|:---|:---|
| **Gmail** | Agent can send the summary via email | Search "Gmail" in tools, use "Send Email" action |
| **Google Calendar** | Agent can schedule a follow-up reminder | Search "Google Calendar", use "Create Event" |
| **Slack** | Agent can post findings to a channel | Search "Slack", use "Send Message" |
| **Notion** | Agent can create a page in your wiki | Search "Notion", use "Create Page" |

**After adding the tool, test with a prompt like:**

```
Research the latest trends in edge computing. Save a summary to
a Google Doc AND email it to me at [your email].
```

**What to watch for:** The agent should autonomously decide to use BOTH the Google Docs tool AND the Gmail tool -- without you telling it the order. It figures out the sequence by itself. That's the autonomy pillar from Class 5 in action.

### Part B: Zapier AI Agents -- The Quick-Win Alternative (10 min)

> "n8n gives you the full agent loop with maximum control. But sometimes you don't need all that -- you just want a simple AI-powered automation that triggers on an event and handles a task. That's where Zapier comes in."

### What is a Zapier AI Agent?

Zapier's AI features let you build **trigger-based automations** where an LLM handles the 'thinking' step. It's less flexible than n8n's full ReAct loop, but it's faster to set up for straightforward tasks.

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/192/904/original/3.jpeg?1777279423)


**Key difference:** In Zapier, the workflow structure is mostly fixed -- the AI handles one step's logic. In n8n, the AI Agent controls the entire flow, choosing tools dynamically.

### Exercise 4.3 -- Build a Zapier AI Automation

**Open [zapier.com](https://zapier.com/) and create a new Zap.**

**Build this automation:**

**Trigger:** Manual trigger (for testing) or "New email in Gmail"

**AI step:** Add a "ChatGPT" action step (Zapier has a built-in OpenAI integration):
- **Prompt:** `Classify the following text into one of these categories: BUG_REPORT, FEATURE_REQUEST, QUESTION, COMPLAINT. Also rate the urgency as LOW, MEDIUM, or HIGH. Respond with JSON only: {"category": "...", "urgency": "..."}`
- **Input:** The email body (or test text)

**Action:** Based on the AI's classification, add a "Send Slack Message" step:
- Route BUG_REPORT to #engineering
- Route FEATURE_REQUEST to #product
- Route COMPLAINT with HIGH urgency to #urgent-support

**What success looks like:** You send a test email, the AI classifies it, and the right Slack channel gets notified -- all without you doing anything after setup.

**Beginner Note:** Zapier's interface is very guided -- it walks you through each step. If you get stuck, the built-in help is excellent.

**Intermediate Note:** This is technically an AI-powered automation, not a full agent (no loop, no dynamic tool selection). But it's the pattern most teams ship first because it's reliable and predictable.

## Section 5 -- Guided Exercise: Error Handling & Retry Logic (20 min)

> "Here's what separates a demo agent from a production agent: **what happens when things go wrong**. And things WILL go wrong. APIs time out, search results are empty, the LLM hallucinates a tool call, rate limits hit. Today we learn the patterns for handling all of these."


### The 5 Failure Modes of Agentic Loops

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/192/905/original/4.jpeg?1777279561)


### Exercise 5.1 -- Add Error Handling to Your n8n Agent

**Go back to your research agent workflow in n8n and add these safeguards:**

**Safeguard 1: Max Iterations**
- Click on your AI Agent node
- Go to **Settings** (gear icon or Options)
- Find **"Max Iterations"** and set it to **10**
- This prevents infinite loops -- after 10 tool calls, the agent stops and returns whatever it has

**Safeguard 2: Retry on Fail**
- Click on each tool node (Web Search, Google Docs)
- Go to **Settings > "Retry on Fail"**
- Set **retries to 3** with **wait between retries of 2000ms** (2 seconds)
- This handles transient API failures automatically

**Safeguard 3: Improve System Message for Edge Cases**
- Add these lines to your agent's system message:

In [ ]:
# Add to your n8n agent's system message:

# ERROR HANDLING RULES:
# - If web search returns no relevant results, tell the user honestly.
#   Do NOT make up information to fill the gap.
# - If a tool call fails after retries, skip that step and explain
#   what happened in your response.
# - If you are unsure which tool to use, prefer web search as a
#   safe default.
# - Maximum 8 tool calls per task. If you haven't finished by then,
#   return what you have and explain what's still missing.
# - Never send emails or create documents unless the user explicitly
#   asked for it.

**Test the error handling by deliberately triggering edge cases:**

**Test 1 -- Empty results:**
```
Research the internal quarterly revenue of a fictional company
called "Nexolith Systems" founded in 2025.
```
The agent should report that it couldn't find information, not make up revenue figures.

**Test 2 -- Ambiguous request:**
```
Research everything about AI.
```
A vague request like this could trigger many tool calls. Your max iterations limit should prevent a runaway loop.

**Test 3 -- No explicit save instruction:**
```
What's the latest news about Kubernetes?
```
The agent should answer in chat without creating a Google Doc (because you didn't ask it to). If it creates a doc anyway, your system message needs the "never create documents unless asked" rule.


**What success looks like:** Your agent handles all three edge cases gracefully -- honest about missing data, stops within iteration limits, and doesn't take actions the user didn't request.

**Advanced Extension:** In production, you'd add a dedicated error-handling workflow that catches agent failures and logs them to a monitoring system. n8n supports error workflows -- you can create a separate workflow that triggers when any node in your main workflow fails.

### Exercise 5.2 -- The Retry Pattern Explained

**Instructor says:**

> "Let me show you the most important production pattern for agent reliability: exponential backoff. It's the same pattern you use for API retries in backend code."

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/192/917/original/5.jpeg?1777283173)



**Beginner Note:** You don't need to implement this yourself in n8n -- the "Retry on Fail" setting does it for you. But understanding the pattern matters because you'll code it by hand in later classes.

**Intermediate Note:** n8n's retry settings let you configure the wait time and number of retries per node. For most tools, 3 retries with 2-second waits is a good default.

## Section 6 -- Mini Build: Production-Grade Research Agent (25 min)

> "You've built the basics and learned error handling. Now let's put it all together into something you'd actually be comfortable running regularly. Your mission: upgrade your research agent into something production-worthy."


### Your Final Agent Spec

Build an n8n workflow that:

```
RESEARCH AGENT -- PRODUCTION SPEC:

  TRIGGER: Chat message with a research topic

  TOOLS:
  ├── Web Search (find information)
  ├── Google Docs (save output)
  └── [Your choice: Gmail, Slack, Calendar, or Notion]

  SYSTEM MESSAGE includes:
  ├── Research methodology (search, analyse, summarise)
  ├── Output format (structured with sections)
  ├── Source citation rules
  ├── Error handling instructions
  └── Boundaries (what NOT to do)

  SAFEGUARDS:
  ├── Max iterations: 10
  ├── Retry on fail: 3 attempts per tool
  ├── Memory: Window buffer (10 messages)
  └── System message guardrails

  SUCCESS CRITERIA:
  ├── Agent searches the web (verified in execution log)
  ├── Summary is structured and cites sources
  ├── Google Doc is created with proper title and content
  ├── Third tool is used when appropriate
  ├── Edge cases handled gracefully (empty results, vague queries)
  └── Agent stays within iteration limits
```


In [ ]:
# Test your production research agent with these prompts:

test_prompts = [
    # Test 1: Standard research task
    "Research the current state of quantum computing in 2026. "
    "Focus on practical applications, not theory. "
    "Save to a Google Doc and email me a summary.",

    # Test 2: Edge case -- niche topic with limited info
    "Research recent academic papers about using AI agents "
    "for automated database schema migration. "
    "Save whatever you find to a Google Doc.",

    # Test 3: Edge case -- overly broad request
    "Tell me everything about machine learning.",
    # (Agent should scope this down, not loop forever)
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"Test {i}:")
    print(f"  {prompt}")
    print()

**After testing, answer these questions for yourself:**

```
SELF-ASSESSMENT:

  [ ] Does your agent search the web in every test? (Check execution log)
  [ ] Are the summaries structured and sourced?
  [ ] Does Google Docs integration work reliably?
  [ ] Does the agent handle the niche topic gracefully (Test 2)?
  [ ] Does the agent scope down the broad request (Test 3)?
  [ ] Does your third tool fire when relevant?
  [ ] Did any test hit the max iteration limit?
  [ ] Would you trust this agent to run without supervision?
```

**Beginner Note:** If you're still stuck on the basics, focus on getting Tests 1 and 2 working. The error handling refinements can come after.


## Section 7 -- Wrap-Up (10 min)


### What we built and learned today:

| Skill | What You Can Do Now |
|:---|:---|
| **n8n AI Agent** | Wire an LLM brain to real tools on a visual canvas and let it run the ReAct loop |
| **Tool design** | Understand what counts as a tool and how the LLM autonomously selects which to invoke |
| **Zapier AI** | Build simple trigger-based AI automations for quick wins |
| **Error handling** | Add max iterations, retry logic, and system-message guardrails to make agents reliable |
| **Production agent** | Build a working research agent that searches, summarises, saves, and handles edge cases |

---

### Key takeaways

1. **Tools are what make agents real.** Without tools, an agent is just a chatbot talking to itself. Today you connected real tools -- web search, Google Docs, Gmail -- and watched the agent use them autonomously.

2. **The agent decides which tool to use.** You didn't hardcode the sequence. The LLM read your goal, looked at its available tools, and figured out the order. That's the fundamental difference between an automation and an agent.

3. **Tool descriptions matter enormously.** The LLM picks tools based on their descriptions. A vague description leads to wrong tool selection. A clear description leads to reliable behaviour.

4. **Error handling is what separates demos from production.** Max iterations prevent infinite loops. Retries handle transient failures. System-message guardrails prevent hallucinated actions. Add all three to every agent you build.

5. **Start with no-code, graduate to code.** n8n and Zapier let you prototype and ship agent workflows fast. When you hit their limits, you'll have the design intuition to build in code -- and we'll cover that in the next classes.

6. **The spec from Class 5 maps directly to n8n nodes.** Your paper spec had: goal (system message), tools (connected nodes), memory (buffer), guardrails (settings). Today you built exactly that.

